#### Average Close and Volume for Each Stock: Calculate the average closing price and volume for each stock. Provide insights into the overall performance and trading activity of each stock. (Trade Data) 
###

In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.window import Window
from pyspark.sql.functions import *


df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

window_spec = Window.partitionBy(col('Name'))
df1 = df.withColumn('avg_close' , round(avg(col('Volume')).over(window_spec),2) )
df2 = df1.withColumn('avg_volume' , round(avg(col('close')).over(window_spec),2) )
df2.show()

### Highest Closing Price of Each Quarter for Each Calendar Year: Identify and extract the highest closing price for each quarter of each calendar year. This will help investors understand the quarterly performance trends. (Trade Data) ###


In [0]:
# df.printSchema()
# print(df.dtypes)
# df['date'].dataType
# df.withColumn('date' , to_date(col('date')) )

from pyspark.sql.functions import *
from pyspark.sql.window import Window

df1 = df.withColumn(
    "Quarter",
     when(month(col("Date")).isin(1,2,3), 1)
    .when(month(col("Date")).isin(4,5,6), 2)
    .when(month(col("Date")).isin(7,8,9), 3)
    .when(month(col("Date")).isin(10,11,12), 4)
)
df1 = df1.withColumn('Year' , year(col('date')))

df2  = df1.groupBy(col('Year') , col('Quarter')).agg(max(col('close')).alias('Highest Closing Price'))

display(df2)

###Bar Graph for Daily Price Gap Analysis: Create a bar graph to visually analyze the daily price gap, which is the difference between the previous day's close and the current day's open. This will provide insights into potential market gaps and their impact on stock prices. For AAL Stock. (Trade Data) ###

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

In [0]:
window_spec = Window.partitionBy().orderBy(col('date'))
df = df.withColumn( 'prev_close' , lag(col('close')).over(window_spec) )
df = df.withColumn('difference', (col('close') - col('prev_close')) )
display(df)

###Identify in which year maximum number of customer become the member and among those which customer did most transaction. (Starbucks Data) 
###

In [0]:
df = spark.read.format('csv').option('inferschema', True).option('header',True).load('/Volumes/workspace/default/assignment_7/profile.csv')

In [0]:
df = df.withColumn('date' , to_date(col('became_member_on') , 'yyyyMMdd')) 
df = df.groupBy(year(col('date'))).agg(count(col('id')).alias('Total Counts'))
display(df)

###Calculate average Price-to-Earnings (P/E) Ratio for each Stock: Create a new column named pe_ratio representing the Price-to-Earnings ratio for each stock on a given day. The P/E ratio is calculated as the closing price divided by the earnings per share (EPS). Assume a constant EPS for simplicity. Use EPS = 5.0. (Trade Data) 
###

In [0]:
df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

In [0]:
df = df.withColumn('earning', round((col('high') - col('low') ),3).alias('Earnings') )
df = df.withColumn('P/E Ratio' , round(( (col('close'))/col('earning') ), 3).alias('EPS') )

display(df)

###Moving Average Trend Analysis: Implement a transformation to calculate the 20-day moving average (ma_20) for the closing prices of each stock. This moving average will be used to identify potential trends and smooth out short-term fluctuations. (Trade Data) 


In [0]:
from pyspark.sql.windows import Window

df = spark.read.format('csv').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

window_spec = Window.partitionBy(col('Name')).orderBy(col('close'))

df = df.withColumn('moving average' , avg(col('close')) over(col('')) )

df.show()

###Moving Average Trend Analysis: Implement a transformation to calculate the 20-day moving average (ma_20) for the closing prices of each stock. This moving average will be used to identify potential trends and smooth out short-term fluctuations. (Trade Data) 

###

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

windowSpec = Window.partitionBy("name").orderBy("date").rowsBetween(-19,0)

ma_df = df.withColumn(
    "ma_20",
    avg("close").over(windowSpec)
)

display(ma_df)

In [0]:
df = df.withColumn(
    "Intraday_Profit",
    abs(col("close") - col("open"))
)

df = df.groupBy(month(col("date")).alias("Month")) \
       .agg(avg("Intraday_Profit").alias("Avg_Intraday_Profit"))

display(df)